# Colab preparation

This cell gets the latest code from GitHub (clone on first run, then `git fetch` + `reset --hard origin/main`), mounts Google Drive, and installs only missing packages. It does not train or draw the figure.

Run this cell after we push updates; it prints the repo commit hash. Colab can still *display* old markdown in cells you opened earlier—trust the printed commit, or reopen the notebook from GitHub. Data: `My Drive/Foundation Model/vendor/pfdelta_task1.1`.


In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
    else:
        subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin"])
        subprocess.check_call(["git", "-C", str(REPO), "reset", "--hard", "origin/main"])
    os.chdir(REPO)
    head = subprocess.check_output(
        ["git", "-C", str(REPO), "log", "-1", "--oneline"], text=True
    ).strip()
    print(f"Repo at {head}", flush=True)
    from google.colab import drive

    drive.mount("/content/drive")

missing = []
for module, package in (
    ("numpy", "numpy"),
    ("torch", "torch"),
    ("matplotlib", "matplotlib"),
    ("pyarrow", "pyarrow"),
    ("huggingface_hub", "huggingface_hub"),
):
    try:
        importlib.import_module(module)
    except ImportError:
        missing.append(package)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("Required packages are already installed.")


# Training

This cell trains **TPGT** only (type-pair graph Transformer, independent weights per stage) on that Drive dataset. It writes checkpoints to `Foundation Model/tpgt_task11_v7` and comparison tables to `Foundation Model/results/tpgt_task11_v7` (the folder name follows `RUN_TAG` in the script, so older runs such as `Foundation Model/tpgt_task11` are never overwritten). It does not draw the figure.

Use a **GPU** runtime. Run the prep cell first so you have the latest `train_tpgt_task11.py` from GitHub. A finished seed (100 epochs, matching checkpoint version) is skipped if this cell is run again; obsolete Drive checkpoints (wrong version, old RPE width, or corrupt files) are ignored and training restarts from epoch 0.

Training uses feasible **N** only (same split as GENCO). **100 epochs**, batch **32**, loss **0.85** state + **0.15** physics, seeds 0–2. Set **`DATA_FRACTION`** in the training cell (e.g. **0.25**) to use the **first** that fraction of scenarios in each train/valid/test folder (fixed parquet order). Raising it to **0.30** keeps the same first 25% and **adds** the next 5%; cached splits extend instead of rebuilding from scratch. **1.0** is the full dataset.

Optional: set `GENCO_TPGT_COMPILE=1` (or pass `--compile` / `--fast` to `%run`) for `torch.compile`; default training already uses batched type-pair attention.

Pair features are a 5-channel **meshed-network RPE** (full topology: \(\mathrm{Re}\,Y_{ij}\), \(\mathrm{Im}\,Y_{ij}\), adjacency, hop distance, effective resistance), not a spanning tree. During training each RPE channel is zeroed with probability 0.1 per step (never at validation/test). Width/depth (48 hidden, 24 pair/attn, 2 heads, 4 correction stages, 3-layer MLPs) are implementation choices, not the authors' hyperparameters.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"
COLAB_DATA = "/content/drive/MyDrive/Foundation Model/vendor/pfdelta_task1.1"
WINDOWS_DATA = Path(r"K:\My Drive\Foundation Model\vendor\pfdelta_task1.1")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
    else:
        subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin"])
        subprocess.check_call(["git", "-C", str(REPO), "reset", "--hard", "origin/main"])
    os.chdir(REPO)
    head = subprocess.check_output(
        ["git", "-C", str(REPO), "log", "-1", "--oneline"], text=True
    ).strip()
    print(f"Training from {head}", flush=True)
else:
    here = Path.cwd()
    if (here / "train_tpgt_task11.py").is_file():
        pass
    elif (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")

if on_colab():
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive

        drive.mount("/content/drive")
    os.environ["GENCO_PFDELTA_TASK11"] = COLAB_DATA
elif WINDOWS_DATA.is_dir():
    os.environ["GENCO_PFDELTA_TASK11"] = str(WINDOWS_DATA)

# 1.0 = full PFΔ split; 0.25 ≈ 12150 train / 1350 valid / 500 test (N) cases. Batch size is always 32. Restart runtime if GPU OOM.
DATA_FRACTION = 1.0
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
print(f"DATA_FRACTION={DATA_FRACTION}", flush=True)

%run train_tpgt_task11.py


# Other PFΔ tasks (IEEE 118)

Each cell below trains TPGT on one PFΔ task with the same model, loss and schedule as Task 1.1. Run the preparation cell once per runtime, then the cell of the task you want (one task per runtime at a time; separate Colab sessions can run different tasks because every task has its own folders). Re-running a cell resumes from that task's last checkpoint.

| Task | Training / validation data | Train scenarios | Drive run folder |
| --- | --- | --- | --- |
| 1.1 | N (cell above) | 48,600 | `tpgt_task11_v7` |
| 1.2 | N + N-1 | 48,600 | `tpgt_task1.2_v7` |
| 1.3 / 2.1 | N + N-1 + N-2 (one dataset, one folder) | 48,600 | `tpgt_task1.3_v7` |
| 2.3 | N + N-1 + N-2, 3x less data | 16,200 | `tpgt_task2.3_v7` |
| 4.1 | 90% feasible + 10% close-to-infeasible (C2I) | 48,600 | `tpgt_task4.1_v7` |
| 4.2 | 50% feasible + 10% C2I + 40% approaching infeasibility | 48,600 | `tpgt_task4.2_v7` |
| 4.3 | 20% C2I + 80% approaching infeasibility | 48,600 | `tpgt_task4.3_v7` |

Every task is tested on the same six folders as Task 1.1 (N, N-1, N-2 feasible and the three C2I sets).

* **Data**: `My Drive/Foundation Model/vendor/pfdelta_task<id>` (Task 2.1 uses `pfdelta_task1.3`). If it is missing or incomplete, the first run downloads `gridfm/pfdelta_task<id>` from Hugging Face into that folder (about 0.3 to 0.7 GB).
* **`DATA_FRACTION`** at the top of each cell keeps the first fraction of every train/valid/test folder, so the task's mix (N / N-1 / N-2, feasible / C2I / approaching) is preserved. Raising it later extends the cached prefixes.
* **First run** builds split caches (a few minutes; N-2 folders hold thousands of topologies). They are mirrored to `<run folder>/cache` on Drive (up to about 4 GB for tasks 1.3, 4.1 and 4.2) and can be deleted at any time; they are rebuilt from the parquet files.
* **Every checkpoint epoch (3, 6, ...)** writes into the run folder: `seed_<s>/epoch_<e>.pt`, the test-PBL tables and the bar plot `results/figure4_task<id>_tpgt.png/.svg` (TPGT mean ± std over the seeds so far vs. the published GENCO Base, CANOS-PF, GNS-S and PFNet numbers for that task), and IEEE 118 bus error maps in `bus_error_maps/seed_<s>/epoch_<e>/`. Each cell shows the latest bar plot and maps when training stops.
* Batches of 32 cases mix topologies; each topology in a batch gets its own forward pass, so tasks with N-1/N-2 training data take longer per epoch than Task 1.1.


In [ ]:
import os
import subprocess
from pathlib import Path

from IPython.display import Image, display

TASK = "1.2"
DATA_FRACTION = 1.0

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
    else:
        subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin"])
        subprocess.check_call(["git", "-C", str(REPO), "reset", "--hard", "origin/main"])
    os.chdir(REPO)
    head = subprocess.check_output(
        ["git", "-C", str(REPO), "log", "-1", "--oneline"], text=True
    ).strip()
    print(f"Training from {head}", flush=True)
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive

        drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive")
else:
    here = Path.cwd()
    if (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    elif not (here / "train_tpgt_task11.py").is_file():
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")
    DRIVE = Path(r"K:\My Drive")

DATA_ID = "1.3" if TASK in ("1.3", "2.1") else TASK
RUN = DRIVE / "Foundation Model" / f"tpgt_task{DATA_ID}_v7"
os.environ["GENCO_PFDELTA_TASK" + DATA_ID.replace(".", "")] = str(
    DRIVE / "Foundation Model" / "vendor" / f"pfdelta_task{DATA_ID}"
)
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
print(f"Task {TASK}: DATA_FRACTION={DATA_FRACTION}, run folder {RUN}", flush=True)

%run train_tpgt_task11.py --task {TASK}

bar = RUN / "results" / f"figure4_task{DATA_ID}_tpgt.png"
if bar.is_file():
    display(Image(filename=str(bar)))
maps = sorted(
    RUN.glob("bus_error_maps/seed_*/epoch_*"),
    key=lambda path: (int(path.name.split("_")[1]), path.parent.name),
)
if maps:
    print(f"Latest bus error maps: {maps[-1]}", flush=True)
    for png in sorted(maps[-1].glob("*.png")):
        display(Image(filename=str(png)))


In [ ]:
import os
import subprocess
from pathlib import Path

from IPython.display import Image, display

TASK = "1.3"
DATA_FRACTION = 1.0

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
    else:
        subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin"])
        subprocess.check_call(["git", "-C", str(REPO), "reset", "--hard", "origin/main"])
    os.chdir(REPO)
    head = subprocess.check_output(
        ["git", "-C", str(REPO), "log", "-1", "--oneline"], text=True
    ).strip()
    print(f"Training from {head}", flush=True)
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive

        drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive")
else:
    here = Path.cwd()
    if (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    elif not (here / "train_tpgt_task11.py").is_file():
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")
    DRIVE = Path(r"K:\My Drive")

DATA_ID = "1.3" if TASK in ("1.3", "2.1") else TASK
RUN = DRIVE / "Foundation Model" / f"tpgt_task{DATA_ID}_v7"
os.environ["GENCO_PFDELTA_TASK" + DATA_ID.replace(".", "")] = str(
    DRIVE / "Foundation Model" / "vendor" / f"pfdelta_task{DATA_ID}"
)
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
print(f"Task {TASK}: DATA_FRACTION={DATA_FRACTION}, run folder {RUN}", flush=True)

%run train_tpgt_task11.py --task {TASK}

bar = RUN / "results" / f"figure4_task{DATA_ID}_tpgt.png"
if bar.is_file():
    display(Image(filename=str(bar)))
maps = sorted(
    RUN.glob("bus_error_maps/seed_*/epoch_*"),
    key=lambda path: (int(path.name.split("_")[1]), path.parent.name),
)
if maps:
    print(f"Latest bus error maps: {maps[-1]}", flush=True)
    for png in sorted(maps[-1].glob("*.png")):
        display(Image(filename=str(png)))


In [ ]:
import os
import subprocess
from pathlib import Path

from IPython.display import Image, display

TASK = "2.3"
DATA_FRACTION = 1.0

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
    else:
        subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin"])
        subprocess.check_call(["git", "-C", str(REPO), "reset", "--hard", "origin/main"])
    os.chdir(REPO)
    head = subprocess.check_output(
        ["git", "-C", str(REPO), "log", "-1", "--oneline"], text=True
    ).strip()
    print(f"Training from {head}", flush=True)
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive

        drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive")
else:
    here = Path.cwd()
    if (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    elif not (here / "train_tpgt_task11.py").is_file():
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")
    DRIVE = Path(r"K:\My Drive")

DATA_ID = "1.3" if TASK in ("1.3", "2.1") else TASK
RUN = DRIVE / "Foundation Model" / f"tpgt_task{DATA_ID}_v7"
os.environ["GENCO_PFDELTA_TASK" + DATA_ID.replace(".", "")] = str(
    DRIVE / "Foundation Model" / "vendor" / f"pfdelta_task{DATA_ID}"
)
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
print(f"Task {TASK}: DATA_FRACTION={DATA_FRACTION}, run folder {RUN}", flush=True)

%run train_tpgt_task11.py --task {TASK}

bar = RUN / "results" / f"figure4_task{DATA_ID}_tpgt.png"
if bar.is_file():
    display(Image(filename=str(bar)))
maps = sorted(
    RUN.glob("bus_error_maps/seed_*/epoch_*"),
    key=lambda path: (int(path.name.split("_")[1]), path.parent.name),
)
if maps:
    print(f"Latest bus error maps: {maps[-1]}", flush=True)
    for png in sorted(maps[-1].glob("*.png")):
        display(Image(filename=str(png)))


In [ ]:
import os
import subprocess
from pathlib import Path

from IPython.display import Image, display

TASK = "4.1"
DATA_FRACTION = 1.0

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
    else:
        subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin"])
        subprocess.check_call(["git", "-C", str(REPO), "reset", "--hard", "origin/main"])
    os.chdir(REPO)
    head = subprocess.check_output(
        ["git", "-C", str(REPO), "log", "-1", "--oneline"], text=True
    ).strip()
    print(f"Training from {head}", flush=True)
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive

        drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive")
else:
    here = Path.cwd()
    if (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    elif not (here / "train_tpgt_task11.py").is_file():
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")
    DRIVE = Path(r"K:\My Drive")

DATA_ID = "1.3" if TASK in ("1.3", "2.1") else TASK
RUN = DRIVE / "Foundation Model" / f"tpgt_task{DATA_ID}_v7"
os.environ["GENCO_PFDELTA_TASK" + DATA_ID.replace(".", "")] = str(
    DRIVE / "Foundation Model" / "vendor" / f"pfdelta_task{DATA_ID}"
)
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
print(f"Task {TASK}: DATA_FRACTION={DATA_FRACTION}, run folder {RUN}", flush=True)

%run train_tpgt_task11.py --task {TASK}

bar = RUN / "results" / f"figure4_task{DATA_ID}_tpgt.png"
if bar.is_file():
    display(Image(filename=str(bar)))
maps = sorted(
    RUN.glob("bus_error_maps/seed_*/epoch_*"),
    key=lambda path: (int(path.name.split("_")[1]), path.parent.name),
)
if maps:
    print(f"Latest bus error maps: {maps[-1]}", flush=True)
    for png in sorted(maps[-1].glob("*.png")):
        display(Image(filename=str(png)))


In [ ]:
import os
import subprocess
from pathlib import Path

from IPython.display import Image, display

TASK = "4.2"
DATA_FRACTION = 1.0

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
    else:
        subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin"])
        subprocess.check_call(["git", "-C", str(REPO), "reset", "--hard", "origin/main"])
    os.chdir(REPO)
    head = subprocess.check_output(
        ["git", "-C", str(REPO), "log", "-1", "--oneline"], text=True
    ).strip()
    print(f"Training from {head}", flush=True)
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive

        drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive")
else:
    here = Path.cwd()
    if (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    elif not (here / "train_tpgt_task11.py").is_file():
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")
    DRIVE = Path(r"K:\My Drive")

DATA_ID = "1.3" if TASK in ("1.3", "2.1") else TASK
RUN = DRIVE / "Foundation Model" / f"tpgt_task{DATA_ID}_v7"
os.environ["GENCO_PFDELTA_TASK" + DATA_ID.replace(".", "")] = str(
    DRIVE / "Foundation Model" / "vendor" / f"pfdelta_task{DATA_ID}"
)
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
print(f"Task {TASK}: DATA_FRACTION={DATA_FRACTION}, run folder {RUN}", flush=True)

%run train_tpgt_task11.py --task {TASK}

bar = RUN / "results" / f"figure4_task{DATA_ID}_tpgt.png"
if bar.is_file():
    display(Image(filename=str(bar)))
maps = sorted(
    RUN.glob("bus_error_maps/seed_*/epoch_*"),
    key=lambda path: (int(path.name.split("_")[1]), path.parent.name),
)
if maps:
    print(f"Latest bus error maps: {maps[-1]}", flush=True)
    for png in sorted(maps[-1].glob("*.png")):
        display(Image(filename=str(png)))


In [ ]:
import os
import subprocess
from pathlib import Path

from IPython.display import Image, display

TASK = "4.3"
DATA_FRACTION = 1.0

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
    else:
        subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin"])
        subprocess.check_call(["git", "-C", str(REPO), "reset", "--hard", "origin/main"])
    os.chdir(REPO)
    head = subprocess.check_output(
        ["git", "-C", str(REPO), "log", "-1", "--oneline"], text=True
    ).strip()
    print(f"Training from {head}", flush=True)
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive

        drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive")
else:
    here = Path.cwd()
    if (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    elif not (here / "train_tpgt_task11.py").is_file():
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")
    DRIVE = Path(r"K:\My Drive")

DATA_ID = "1.3" if TASK in ("1.3", "2.1") else TASK
RUN = DRIVE / "Foundation Model" / f"tpgt_task{DATA_ID}_v7"
os.environ["GENCO_PFDELTA_TASK" + DATA_ID.replace(".", "")] = str(
    DRIVE / "Foundation Model" / "vendor" / f"pfdelta_task{DATA_ID}"
)
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
print(f"Task {TASK}: DATA_FRACTION={DATA_FRACTION}, run folder {RUN}", flush=True)

%run train_tpgt_task11.py --task {TASK}

bar = RUN / "results" / f"figure4_task{DATA_ID}_tpgt.png"
if bar.is_file():
    display(Image(filename=str(bar)))
maps = sorted(
    RUN.glob("bus_error_maps/seed_*/epoch_*"),
    key=lambda path: (int(path.name.split("_")[1]), path.parent.name),
)
if maps:
    print(f"Latest bus error maps: {maps[-1]}", flush=True)
    for png in sorted(maps[-1].glob("*.png")):
        display(Image(filename=str(png)))


# Diagnostics and bus error maps

Run these after (or between) training sessions; they cannot run while the training cell is running. Both use the latest checkpoint of the run folder of `TASK` (set at the top of each cell; `"1.1"` is `Foundation Model/tpgt_task11_v7`, other tasks `Foundation Model/tpgt_task<id>_v7`) and the split caches the training cell built.

**Diagnose** prints: oracle / flat-start / per-stage PBL, state RMSE by bus type, RPE channel zero-ablation, RPE and residual correctness audits, and the N-1 / N-2 contingency report split into line vs generator outages.

**Bus error maps** draw per-bus PBL on the IEEE 118 one-line layout (validation N, worst generator outage, worst and median line outage), plus a hop-distance table, and display them below.


In [ ]:
import os
import subprocess
from pathlib import Path

TASK = "1.1"

REPO = Path("/content/genco-figure4-task11")
subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin"])
subprocess.check_call(["git", "-C", str(REPO), "reset", "--hard", "origin/main"])
os.chdir(REPO)
if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive

    drive.mount("/content/drive")
DATA_ID = "1.3" if TASK in ("1.3", "2.1") else TASK
os.environ["GENCO_PFDELTA_TASK" + DATA_ID.replace(".", "")] = (
    f"/content/drive/MyDrive/Foundation Model/vendor/pfdelta_task{DATA_ID}"
)
os.environ["GENCO_TPGT_DATA_FRACTION"] = "1.0"

%run train_tpgt_task11.py --diagnose --task {TASK} --seed 0 --cases 32


In [ ]:
import os
from pathlib import Path

from IPython.display import Image, display

TASK = "1.1"

os.chdir("/content/genco-figure4-task11")
DATA_ID = "1.3" if TASK in ("1.3", "2.1") else TASK
RUN_NAME = "tpgt_task11_v7" if DATA_ID == "1.1" else f"tpgt_task{DATA_ID}_v7"
RUN = Path("/content/drive/MyDrive/Foundation Model") / RUN_NAME
SEED = 0
CHECKPOINT = RUN / f"seed_{SEED}" / "last.pt"
CACHE = Path(f"/content/{RUN_NAME}_cache")
if not (CACHE / "test_n-1_feasible.pt").is_file():
    CACHE = RUN / "cache"
VAL_CACHE = next(
    name
    for name in ("valid_n_feasible.pt", "valid_n_around_nose.pt", "valid_n_nose.pt")
    if (CACHE / name).is_file()
)
OUT = RUN / "bus_error_maps" / f"seed_{SEED}"

%run plot_bus_error_map.py --checkpoint "{CHECKPOINT}" --cache-dir "{CACHE}" --val-cache "{VAL_CACHE}" --out-dir "{OUT}"

for name in (
    "val_N_mean_pbl.png",
    "val_N_dP_dQ.png",
    "test_N1_gen_outage_worst.png",
    "test_N1_line_outage_worst.png",
    "test_N1_line_outage_median.png",
):
    if (OUT / name).is_file():
        display(Image(filename=str(OUT / name)))


# SVG figure

This cell draws the published Task 1.1 SVG (GENCO Base, CANOS-PF, GNS-S, PFNet) from the published table. It does not train.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


here = Path.cwd()
if not (here / "plot_task1_1.py").is_file():
    if on_colab():
        if not REPO.is_dir():
            subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
        os.chdir(REPO)
    elif (here / "genco" / "plot_task1_1.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("plot_task1_1.py is not in this folder.")

%run plot_task1_1.py --published
